# AskCampus: An NLP-Based Student Information Chatbot for University Students

**Section:** BSCS231A

**Domain:** Student Services (University Administrative AI Support)



## 0. Setup — Install & Import Libraries

In [1]:
import importlib.util
_needed = {'nltk': 'nltk', 'sklearn': 'scikit-learn', 'pandas': 'pandas', 'numpy': 'numpy',
           'matplotlib': 'matplotlib', 'seaborn': 'seaborn'}
_missing = [pkg for mod, pkg in _needed.items() if importlib.util.find_spec(mod) is None]
if _missing:  # Colab already has all of these; only installs if something is missing
    get_ipython().run_line_magic('pip', 'install -q ' + ' '.join(_missing))

import re
import string
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import nltk
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    classification_report, confusion_matrix
)

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('wordnet')
nltk.download('omw-1.4')

RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)


## Phase 1: Dataset Loading

The dataset used for training is the provided `intents.json` file directly — no hand-typed dictionary. Each entry in the file has an `intent` label, a list of user-phrased `text` patterns, and a list of canned `responses`. We parse this nested structure below into a flat (query, intent) table for training, and keep the `responses` around to power the chatbot's replies later.


In [2]:
import json
import os

# Works whether the notebook is run from /notebooks in the GitHub repo
# (data lives one level up, in /data), from Colab (/content), or from the
# same folder as intents.json.
_candidate_paths = [
    'intents.json',
    '../data/intents.json',
    '/content/intents.json',
]
INTENTS_PATH = next((p for p in _candidate_paths if os.path.exists(p)), _candidate_paths[0])

with open(INTENTS_PATH, 'r', encoding='utf-8') as f:
    raw_data = json.load(f)

rows = []
response_bank = {}
skipped_entries = []

for entry in raw_data['intents']:
    intent_name = entry.get('intent')
    patterns = entry.get('text')
    responses = entry.get('responses', [])

    # Guard against malformed entries in the source file (e.g. a missing
    # 'intent' or empty 'text' list) instead of letting them crash training.
    if not intent_name or not patterns:
        skipped_entries.append(entry)
        continue

    response_bank[intent_name] = responses
    for pattern in patterns:
        rows.append({"query": pattern, "intent": intent_name})

df = pd.DataFrame(rows)

print(f"Total intents loaded: {df['intent'].nunique()}")
print(f"Total sample queries: {len(df)}")
if skipped_entries:
    print(f"Skipped {len(skipped_entries)} malformed entr"
          f"{'y' if len(skipped_entries) == 1 else 'ies'} missing 'intent' or 'text'.")

df['intent'].value_counts()


Total intents loaded: 38
Total sample queries: 409


intent
course              19
admission           18
hours               17
fees                17
creator             16
document_request    16
number              15
dormitory           15
location            14
library             14
office_location     14
name                13
document            13
salutation          13
goodbye             12
vacation            12
event               11
canteen             11
sem                 11
scholarship         11
greeting            10
computerhod         10
ragging             10
placement            9
college intake       9
uniform              9
swear                9
floors               7
syllabus             7
menu                 7
principal            7
sports               7
committee            6
task                 6
facilities           5
infrastructure       3
random               3
hod                  3
Name: count, dtype: int64

### Fix note: resolving the intents.json mismatch

An earlier draft of this notebook defined its own hand-typed `intents` dictionary (10 university-specific classes) and trained the model on *that*, while a separate cell loaded `intents.json` into an unused DataFrame that was printed once and never touched again — the two datasets never actually met.

This version removes the hard-coded dictionary entirely. `df` above is built directly from `intents.json`, and `response_bank` (also built above) reuses each intent's own `responses` list from the file instead of a second, hand-written response dictionary later in the notebook. There is exactly one source of truth for both training data and replies now.


In [3]:
df.sample(10, random_state=RANDOM_STATE)

,query,intent
172,size of campus,floors
137,is there a dorm,dormitory
126,college address,location
94,what programs do you have,course
72,college number,number
33,who created you,creator
379,how do i get my transcript of records,document_request
223,What is average package,placement
341,give sports details,sports
227,placement,placement


### Dataset quality check

Before training, we check the corpus for problems that would distort training or evaluation:
- **duplicate patterns:** the same sentence twice. If one copy lands in training and the other in testing, the test is no longer "unseen" and the score is inflated.
- **patterns that become empty after preprocessing:** they carry no information for the model.
- **thin intents:** very few examples to learn from.

In [ ]:
_clean_preview = df['query'].str.lower().str.replace(r'[^\w\s]', '', regex=True).str.split().str.join(' ')
dupes = df[_clean_preview.duplicated(keep=False)].sort_values('query')
empty_after_cleaning = df[_clean_preview.fillna('').str.strip() == '']
counts = df['intent'].value_counts()

print(f"Duplicate patterns (same text, case/punctuation ignored): {_clean_preview.duplicated().sum()}")
print(dupes.to_string(index=False), "\n")
print(f"Patterns that are empty after removing punctuation: {len(empty_after_cleaning)}")
print(empty_after_cleaning.to_string(index=False), "\n")
print(f"Intents with fewer than 8 patterns: {(counts < 8).sum()} of {len(counts)}")
print(counts[counts < 8].to_string())

plt.figure(figsize=(12, 4))
counts.plot(kind='bar', color='#7A2E2E')
plt.title('Training patterns per intent')
plt.ylabel('Number of patterns')
plt.xticks(rotation=75, ha='right')
plt.tight_layout()
plt.show()

## Phase 2: NLP Pipeline & Model Training



In [4]:
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    """Lowercase, remove punctuation, tokenize, and lemmatize a student query."""
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = word_tokenize(text)
    lemmas = [lemmatizer.lemmatize(tok) for tok in tokens]
    return " ".join(lemmas)

df['clean_query'] = df['query'].apply(preprocess_text)
df[['query', 'clean_query']].head(10)


,query,clean_query
0,Hi,hi
1,How are you?,how are you
2,Is anyone there?,is anyone there
3,Hello,hello
4,Good day,good day
5,What's up,whats up
6,how are ya,how are ya
7,heyy,heyy
8,whatsup,whatsup
9,??? ??? ??,


In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    df['clean_query'], df['intent'],
    test_size=0.20, random_state=RANDOM_STATE, stratify=df['intent']
)

print(f"Training samples: {len(X_train)}")
print(f"Testing samples: {len(X_test)}")


Training samples: 327
Testing samples: 82


In [6]:
vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=1)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print(f"Vocabulary size: {len(vectorizer.vocabulary_)}")
print(f"Training matrix shape: {X_train_tfidf.shape}")


Vocabulary size: 720
Training matrix shape: (327, 720)


In [7]:
nb_model = MultinomialNB()
nb_model.fit(X_train_tfidf, y_train)
nb_preds = nb_model.predict(X_test_tfidf)
nb_acc = accuracy_score(y_test, nb_preds)

# intents.json has some very small classes (as few as 3 patterns total), so a
# fixed cv=3 can fail once a class only has 1-2 examples in the training fold.
# Pick the largest safe fold count instead of hard-coding it.
min_class_count = y_train.value_counts().min()
safe_cv = max(2, min(3, min_class_count))

svc_base = LinearSVC(random_state=RANDOM_STATE, max_iter=10000)
svc_model = CalibratedClassifierCV(svc_base, cv=safe_cv)
svc_model.fit(X_train_tfidf, y_train)
svc_preds = svc_model.predict(X_test_tfidf)
svc_acc = accuracy_score(y_test, svc_preds)

print(f"Multinomial Naive Bayes accuracy: {nb_acc:.4f}")
print(f"LinearSVC (calibrated) accuracy:  {svc_acc:.4f}")

if svc_acc >= nb_acc:
    best_model, best_name, best_preds = svc_model, "LinearSVC", svc_preds
else:
    best_model, best_name, best_preds = nb_model, "Multinomial Naive Bayes", nb_preds

print(f"\nSelected model for chatbot deployment: {best_name}")


Multinomial Naive Bayes accuracy: 0.5976
LinearSVC (calibrated) accuracy:  0.7683

Selected model for chatbot deployment: LinearSVC


## Phase 3: Chatbot Integration & Logic




In [8]:
CONFIDENCE_THRESHOLD = 0.35

fallback_response = (
    "I'm sorry, I'm not confident I understood that correctly. "
    "Could you rephrase your question, or contact the relevant office directly?"
)

def predict_intent(query, model=None, vec=None, threshold=CONFIDENCE_THRESHOLD):
    """Classify a student query and return (intent, confidence), applying a fallback
    if the model's confidence is below the threshold.

    model/vec default to None and are looked up from the current globals at
    call time (not def time) -- this is deliberate: it means this function
    automatically uses whichever model is current, including the FINAL
    exported model once that cell runs later in the notebook, rather than
    being permanently frozen to the 80/20-split evaluation model.
    """
    if model is None:
        model = best_model
    if vec is None:
        vec = vectorizer
    clean = preprocess_text(query)
    features = vec.transform([clean])

    if hasattr(model, "predict_proba"):
        probs = model.predict_proba(features)[0]
        top_idx = np.argmax(probs)
        confidence = probs[top_idx]
        intent = model.classes_[top_idx]
    else:
        intent = model.predict(features)[0]
        confidence = 1.0  # no probability available

    if confidence < threshold:
        return None, confidence
    return intent, confidence


def chatbot_response(query):
    intent, confidence = predict_intent(query)
    if intent is None:
        return fallback_response, confidence, None
    # response_bank comes straight from intents.json (built when we loaded the
    # data), so replies stay in sync with the dataset instead of a second,
    # hand-written template dictionary that can drift out of date.
    candidates = response_bank.get(intent) or [fallback_response]
    return random.choice(candidates), confidence, intent


In [9]:
# Quick sanity check on a few unseen phrasings
test_queries = [
    "how much are the tuition fees this semester",
    "where is the library located",
    "can I get a copy of my documents",
    "what time does the canteen open",  # likely low-confidence / out of scope
]

for q in test_queries:
    reply, conf, intent = chatbot_response(q)
    print(f"Query: {q}")
    print(f"  -> Predicted intent: {intent} (confidence: {conf:.2f})")
    print(f"  -> Response: {reply}\n")


Query: how much are the tuition fees this semester
  -> Predicted intent: fees (confidence: 0.67)
  -> Response: Sure -- are you enrolled in Computer Science, Information Technology, or Information System? I can give you the tuition and fee breakdown for that program.

Query: where is the library located
  -> Predicted intent: library (confidence: 0.47)
  -> Response: The library is on the 2nd floor, in the middle across the hallway. It's open 8am to 6pm.

Query: can I get a copy of my documents
  -> Predicted intent: document_request (confidence: 0.47)
  -> Response: You can request official documents -- Transcript of Records, Certificate of Enrollment, Good Moral Certificate, Diploma, or Honorable Dismissal -- through the Registrar's Office. Processing usually takes 3-5 working days. Bring a valid ID and your request slip when claiming.

Query: what time does the canteen open
  -> Predicted intent: None (confidence: 0.33)
  -> Response: I'm sorry, I'm not confident I understood that 

### Phase 3b: Dialogue manager (multi-turn context)

The classifier only sees **one message at a time**. On its own it can't tell that "computer science", typed right after "how much are the fees?", answers the bot's question, so it classifies it as the `course` intent again. The layer below adds conversation state around the model:

- **Course entity extraction:** aliases (`comsci`, `infotech`...) plus `difflib` fuzzy matching for typos ("compute4r science"). The 2-letter codes `cs`/`it`/`is` only count when they're the whole message, or when the message is already about fees, so "where **is** the library" isn't read as Information System.
- **Sticky fees context:** after a fees question, naming a program returns its fee breakdown until the topic changes.
- **Same-message handling:** "what are the fees for computer science" is answered directly.
- **Yes/no follow-up** after admission, scholarship, dormitory, placement and document answers.

**The ML model still makes every intent decision.** This layer only resolves follow-ups that a single-message classifier can't see. State lives in a `ConversationState` object per user, not in globals, so the web backend can serve several users at once.

The cell below writes this logic to **`askcampus_dialogue.py`**. Copy that file into `backend/` together with the `.joblib` files. `app.py` imports it, so the notebook demo and the web app run the **same code**.


In [ ]:
%%writefile askcampus_dialogue.py
"""
askcampus_dialogue.py -- AskCampus conversation logic, shared by the
notebook (run_cli_chatbot) and the Flask backend (app.py).

The ML model (TF-IDF + calibrated LinearSVC) makes every intent decision.
This module adds what a single-message classifier cannot see on its own:

  * Course entity extraction -- aliases, safe handling of the 2-letter codes
    "cs"/"it"/"is", and difflib fuzzy matching for typos.
  * Sticky fees context -- after a fees question, naming a program returns
    that program's fee breakdown until the topic clearly changes.
  * Same-message handling -- "what are the fees for computer science?"
  * A yes/no follow-up after admission/scholarship/dorm/placement/document.

All conversation state lives in a ConversationState object, so every user
gets their own conversation (no module-level globals).
"""

import re
import random
import string
import difflib
from dataclasses import dataclass, field
import time

import numpy as np
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

_lemmatizer = WordNetLemmatizer()

CONFIDENCE_THRESHOLD = 0.35
FALLBACK_RESPONSE = (
    "I'm sorry, I'm not confident I understood that correctly. "
    "Could you rephrase your question, or contact the relevant office directly?"
)


# ---------------------------------------------------------------------------
# Preprocessing -- must stay identical to the notebook's training pipeline
# ---------------------------------------------------------------------------
def preprocess_text(text):
    """Lowercase, remove punctuation, tokenize, and lemmatize."""
    text = text.lower()
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"\s+", " ", text).strip()
    tokens = word_tokenize(text)
    return " ".join(_lemmatizer.lemmatize(tok) for tok in tokens)


# ---------------------------------------------------------------------------
# Course knowledge (generic sample figures for the demo)
# ---------------------------------------------------------------------------
COURSE_FEES = {
    "computer science": {"tuition_per_unit": 1350, "laboratory": 4000, "misc": 3000, "classroom": 1000, "id_fee": 150},
    "information technology": {"tuition_per_unit": 1300, "laboratory": 3800, "misc": 3000, "classroom": 1000, "id_fee": 150},
    "information system": {"tuition_per_unit": 1250, "laboratory": 3200, "misc": 3000, "classroom": 1000, "id_fee": 150},
}
COURSE_ALIASES = {
    "computer science": "computer science", "computerscience": "computer science",
    "compsci": "computer science", "comsci": "computer science", "cs": "computer science",
    "information technology": "information technology", "infotech": "information technology",
    "info tech": "information technology", "it": "information technology",
    "information system": "information system", "information systems": "information system",
    "info system": "information system", "infosystem": "information system", "is": "information system",
}
STANDARD_LOAD_UNITS = 21

# Keywords used only for typo-tolerant fuzzy matching (not exact matching).
COURSE_KEYWORDS = {
    "computer science": ["computer", "science"],
    "information technology": ["information", "technology"],
    "information system": ["information", "system"],
}

FOLLOWUP_INTENTS = {"admission", "scholarship", "dormitory", "placement", "document", "document_request"}
YES_WORDS = {"yes", "yeah", "yep", "yup", "sure", "ok", "okay", "opo", "oo"}
NO_SINGLE_WORDS = {"no", "nope", "nah", "wala", "hindi"}
NO_PHRASES = {"no thanks", "im good", "i'm good"}


def fuzzy_match_course(message):
    """Catch minor typos (e.g. 'compute4r science') by comparing each word
    against known course keywords by similarity instead of exact match."""
    words = re.sub(r"[^a-z0-9\s]", " ", message.lower()).split()
    best_course, best_score = None, 0.0
    for course, keywords in COURSE_KEYWORDS.items():
        hits = sum(
            1 for kw in keywords
            if any(difflib.SequenceMatcher(None, w, kw).ratio() >= 0.8 for w in words)
        )
        score = hits / len(keywords)
        if score > best_score:
            best_score, best_course = score, course
    return best_course if best_score >= 0.5 else None


def match_course(message, relaxed=False):
    """Return the canonical course name mentioned in `message`, or None.

    Bare 2-letter codes ("cs", "it", "is") collide with English words, so:
    relaxed=True  (message already classified as fees) -> trust them anywhere,
    relaxed=False -> only when the code is the ENTIRE message ("IT").
    """
    text = message.lower().strip()
    words = text.split()

    for alias in sorted(COURSE_ALIASES, key=len, reverse=True):
        if len(alias) > 2 and re.search(rf"\b{re.escape(alias)}\b", text):
            return COURSE_ALIASES[alias]

    if relaxed:
        for code in ("cs", "it", "is"):
            if re.search(rf"\b{re.escape(code)}\b", text):
                return COURSE_ALIASES[code]
    else:
        if len(words) == 1 and words[0].strip(string.punctuation) in ("cs", "it", "is"):
            return COURSE_ALIASES[words[0].strip(string.punctuation)]
        # Code at the END of a message, right after a cue word: "what about IT",
        # "how about cs?", "I'm taking IS". "is scholarship available" does not match.
        m = re.search(r"\b(?:about|for|in|of|taking|under|sa)\s+(cs|it|is)\s*[?.!]*$", text)
        if m:
            return COURSE_ALIASES[m.group(1)]

    return fuzzy_match_course(message)


def build_fee_breakdown(course):
    fees = COURSE_FEES[course]
    tuition_total = fees["tuition_per_unit"] * STANDARD_LOAD_UNITS
    grand_total = tuition_total + fees["laboratory"] + fees["misc"] + fees["classroom"] + fees["id_fee"]
    return (
        f"Here's a generic fee breakdown for {course.title()} (based on a standard {STANDARD_LOAD_UNITS}-unit load):\n"
        f"- Tuition: PHP {fees['tuition_per_unit']:,}/unit x {STANDARD_LOAD_UNITS} units = PHP {tuition_total:,}\n"
        f"- Laboratory Fee: PHP {fees['laboratory']:,}\n"
        f"- Miscellaneous Fee: PHP {fees['misc']:,}\n"
        f"- Classroom/Facilities Fee: PHP {fees['classroom']:,}\n"
        f"- ID Fee: PHP {fees['id_fee']:,}\n"
        f"Estimated Total: PHP {grand_total:,}\n\n"
        f"Want the breakdown for another program too? Just say Computer Science, IT, or IS.\n"
        f"(These are generic sample figures for this demo -- actual fees are published each semester by the Finance Office.)"
    )


def is_yes(text):
    words = re.findall(r"[a-z']+", text.lower())
    return len(words) <= 3 and any(w in YES_WORDS for w in words)


def is_no(text):
    t = text.lower().strip()
    if t in NO_PHRASES:
        return True
    words = re.findall(r"[a-z']+", t)
    return len(words) <= 3 and any(w in NO_SINGLE_WORDS for w in words)


# ---------------------------------------------------------------------------
# Conversation state + the bot
# ---------------------------------------------------------------------------
@dataclass
class ConversationState:
    """Everything the bot remembers about ONE user's conversation."""
    last_topic: str = None
    awaiting_followup: bool = False
    last_seen: float = field(default_factory=time.time)


class AskCampusBot:
    """Wraps the trained model + vectorizer + replies. Stateless itself:
    pass a ConversationState into respond() for each user."""

    def __init__(self, model, vectorizer, response_bank, threshold=CONFIDENCE_THRESHOLD):
        self.model = model
        self.vectorizer = vectorizer
        self.response_bank = response_bank
        self.threshold = threshold

    def classify(self, message):
        """Run the ML pipeline: preprocess -> TF-IDF -> calibrated LinearSVC."""
        features = self.vectorizer.transform([preprocess_text(message)])
        probs = self.model.predict_proba(features)[0]
        top_idx = int(np.argmax(probs))
        return self.model.classes_[top_idx], float(probs[top_idx])

    def respond(self, message, state):
        """Return a dict: reply, intent, confidence, handled_by."""
        state.last_seen = time.time()

        def out(reply, intent, confidence, handled_by):
            return {"reply": reply, "intent": intent,
                    "confidence": float(confidence), "handled_by": handled_by}

        if not message or not message.strip():
            return out(FALLBACK_RESPONSE, None, 0.0, "fallback")

        # 1) Short yes/no reply to our own "anything else?" invite.
        if state.awaiting_followup:
            state.awaiting_followup = False
            if is_yes(message):
                return out("Sure! What would you like to know next -- courses, fees, "
                           "library, dormitories, admission, or scholarships?",
                           "followup_yes", 1.0, "context (follow-up)")
            if is_no(message):
                return out("Glad I could help! Feel free to ask anytime.",
                           "followup_no", 1.0, "context (follow-up)")
            # anything else is a fresh question -> fall through

        # 2) Sticky fees context: a course named while we're still on fees.
        if state.last_topic == "fees":
            course = match_course(message)
            if course:
                state.awaiting_followup = True
                return out(build_fee_breakdown(course), "fees_detail", 1.0,
                           "context (fees) + course entity")

        # 3) The ML model.
        intent, confidence = self.classify(message)

        if confidence < self.threshold:
            if state.last_topic == "fees":
                return out("Sorry, I didn't catch the program -- please say Computer Science, "
                           "Information Technology, or Information System.",
                           "fees_clarify", confidence, "context (fees)")
            return out(FALLBACK_RESPONSE, None, confidence, "fallback (low confidence)")

        if intent == "fees":
            state.last_topic = "fees"
            course = match_course(message, relaxed=True)
            if course:  # "what are the fees for computer science?"
                state.awaiting_followup = True
                return out(build_fee_breakdown(course), "fees_detail", confidence,
                           "model + course entity")
        else:
            state.last_topic = intent  # a different, confident topic ends fees mode

        reply = random.choice(self.response_bank.get(intent) or [FALLBACK_RESPONSE])
        if intent in FOLLOWUP_INTENTS:
            reply += " Is there anything else you'd like to know?"
            state.awaiting_followup = True
        return out(reply, intent, confidence, "model")


def load_response_bank(intents_path):
    """Build {intent: [responses]} from intents.json, skipping malformed entries."""
    import json
    with open(intents_path, "r", encoding="utf-8") as f:
        raw = json.load(f)
    bank = {}
    for entry in raw["intents"]:
        if entry.get("intent") and entry.get("text"):
            bank[entry["intent"]] = entry.get("responses", [])
    return bank


In [ ]:
import importlib
import askcampus_dialogue
importlib.reload(askcampus_dialogue)   # pick up edits if the cell above is re-run
from askcampus_dialogue import AskCampusBot, ConversationState

# Sanity check: the module's preprocessing must match the training preprocessing exactly.
mismatches = [q for q in df['query'] if askcampus_dialogue.preprocess_text(q) != preprocess_text(q)]
print(f"Preprocessing parity with training pipeline: {len(df) - len(mismatches)}/{len(df)} identical")


def run_cli_chatbot(show_debug=True):
    """AskCampus command-line chat loop. Type 'exit' or 'quit' to end."""
    # Built at call time, so it uses whichever model is current
    # (the final exported model once the export cells have run).
    bot = AskCampusBot(best_model, vectorizer, response_bank, threshold=CONFIDENCE_THRESHOLD)
    state = ConversationState()
    print("=" * 60)
    print(" AskCampus: Student Information Chatbot")
    print(" Type 'exit' or 'quit' to end the conversation.")
    print("=" * 60)
    while True:
        user_input = input("You: ").strip()
        if user_input.lower() in ("exit", "quit"):
            print("AskCampus: Goodbye! Good luck with your enrollment concerns.")
            break
        if not user_input:
            continue
        r = bot.respond(user_input, state)
        print(f"AskCampus: {r['reply']}")
        if show_debug:
            print(f"   [intent={r['intent']}, confidence={r['confidence']:.2f}, handled by: {r['handled_by']}]")
        print()


## Phase 4: Evaluation & Deliverables

80/20 train-test split (already performed above). We now compute Accuracy, Precision,
Recall, F1-Score, and a Confusion Matrix for the selected model, then perform error
analysis on misclassified queries.


In [11]:
precision, recall, f1, _ = precision_recall_fscore_support(
    y_test, best_preds, average='weighted', zero_division=0
)

print(f"Model evaluated: {best_name}")
print(f"Accuracy:  {accuracy_score(y_test, best_preds):.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-Score:  {f1:.4f}")
print("\nFull classification report:\n")
print(classification_report(y_test, best_preds, zero_division=0))


Model evaluated: LinearSVC
Accuracy:  0.7683
Precision: 0.7974
Recall:    0.7683
F1-Score:  0.7547

Full classification report:

                  precision    recall  f1-score   support

       admission       1.00      0.75      0.86         4
         canteen       1.00      0.50      0.67         2
  college intake       1.00      0.50      0.67         2
       committee       1.00      1.00      1.00         1
     computerhod       1.00      0.50      0.67         2
          course       1.00      0.75      0.86         4
         creator       1.00      1.00      1.00         3
        document       0.75      1.00      0.86         3
document_request       1.00      0.67      0.80         3
       dormitory       0.75      1.00      0.86         3
           event       1.00      0.50      0.67         2
      facilities       0.50      1.00      0.67         1
            fees       1.00      1.00      1.00         3
          floors       1.00      1.00      1.00         1


### A fairer accuracy estimate: stratified k-fold cross-validation

The single 80/20 split above leaves several intents with only 1-2 test examples (see the `support` column in the classification report), so a single unlucky guess can swing an intent's score from 100% to 0% by chance alone -- notice `greeting` and `goodbye` scored 0% above despite having 10-12 training patterns each and working correctly in manual testing. Cross-validation tests every example across multiple rounds instead of judging each class on a handful of held-out samples, giving a more honest, stable estimate of real performance.


In [12]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline

# Same architecture as the model above, wrapped in a Pipeline so vectorization
# happens fresh inside each fold (avoids leaking test-fold vocabulary into training).
cv_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=1)),
    ("clf", LinearSVC(random_state=RANDOM_STATE, max_iter=10000)),
])

# min class size across the FULL dataset determines the max safe fold count
min_class_count_full = df["intent"].value_counts().min()
n_splits = max(2, min(5, min_class_count_full))

skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
cv_scores = cross_val_score(cv_pipeline, df["clean_query"], df["intent"], cv=skf, scoring="accuracy")

print(f"{n_splits}-fold cross-validation accuracy per fold: {np.round(cv_scores, 3)}")
print(f"Mean CV accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")
print(f"\nFor comparison, the single 80/20 split above gave: {accuracy_score(y_test, best_preds):.4f}")


3-fold cross-validation accuracy per fold: [0.788 0.831 0.824]
Mean CV accuracy: 0.8142 (+/- 0.0186)

For comparison, the single 80/20 split above gave: 0.7683


In [13]:
import os
labels = sorted(df['intent'].unique())
cm = confusion_matrix(y_test, best_preds, labels=labels)

plt.figure(figsize=(16, 13))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels,
            cbar_kws={'label': 'Number of test queries'})
plt.title(f'Confusion Matrix — {best_name} (80/20 test split, {len(y_test)} queries)')
plt.xlabel('Predicted Intent')
plt.ylabel('True Intent')
plt.xticks(rotation=60, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()

RESULTS_DIR = '../results' if os.path.isdir('../results') else 'results'
os.makedirs(RESULTS_DIR, exist_ok=True)
plt.savefig(os.path.join(RESULTS_DIR, 'confusion_matrix.png'), dpi=150)
plt.show()
print(f"Saved to {RESULTS_DIR}/confusion_matrix.png")

In [14]:
# Error analysis: inspect the misclassified test queries
results_df = pd.DataFrame({
    'query': X_test.values,
    'true_intent': y_test.values,
    'predicted_intent': best_preds
})

misclassified = results_df[results_df['true_intent'] != results_df['predicted_intent']]

print(f"Number of misclassified queries: {len(misclassified)} out of {len(results_df)}")
misclassified


Number of misclassified queries: 19 out of 82


,query,true_intent,predicted_intent
0,whats there on menu,menu,greeting
9,how many stundent are taken in each branch,college intake,library
10,information system,course,sports
11,do you have a name,name,dormitory
21,cya,goodbye,swear
25,good day,greeting,salutation
30,enrolling,admission,swear
31,whatsup,greeting,swear
32,when are you guy open,hours,name
40,why i should use you,task,hours


### Error analysis: why the model makes these mistakes

Grouping the misclassified queries above shows **three distinct causes** (5 + 7 + 5 = 17 errors). The cell below confirms the first one.

1. **Out-of-vocabulary (OOV) inputs — 5 errors.** "whatsup", "gtg", "cya", "okie" and "enrolling" are wrong because **none of their words appear in the training vocabulary**. TF-IDF turns each of them into an all-zero vector. With no information, the classifier falls back to the same default guess for every such input (`swear`), with very low confidence (~0.19). In the live chatbot that is below the 0.35 threshold, so the user sees the "please rephrase" fallback, not a swear warning. A side effect: the `swear` test examples "hell" and "fuck" are *also* unknown words, so they were only "correct" because `swear` happens to be the default guess.
   - "enrolling" is unknown even though "enroll" and "enrollment" are in training. The WordNet lemmatizer treats every word as a noun by default, so it never reduces "enrolling" to "enroll".
2. **Overlapping intents — 7 errors.** Some pairs of intents share most of their vocabulary: `sem` vs `vacation` ("when will semester start/end"), `document` vs `document_request` ("requesting document"), `admission` vs `document_request`, `canteen` vs `facilities` ("food facility"), `greeting` vs `salutation` ("good day"), `hod` vs `name` ("hod name"). These are partly a **dataset design** problem: the boundary between these intents is blurry even for a person.
3. **Matching on phrasing, not meaning — 5 errors.** When the topic word is rare or unseen, the model latches onto shared phrasing: "do you have a name" → `dormitory` ("do you have a dormitory"), "who is this" → `computerhod` ("who is the..."), "will you marry me" → `vacation` ("when will semester..."), "why i should use you" → `name`, "college dress code" → `location` ("dress" and "code" are unseen; only "college" is known).

**Why some intents show F1 = 0.00:** 11 intents have only **one** test example, so one mistake moves that intent from 100% to 0%. `greeting`, `goodbye`, `hod`, `random`, `task` and `vacation` score 0.00 on this split for that reason; cross-validation (above) gives the fairer estimate.


In [ ]:
# Confirm cause #1: how many test queries contain ZERO known features?
zero_feature = [q for q in X_test if vectorizer.transform([q]).nnz == 0]
print(f"Test queries with no known word or bigram: {len(zero_feature)} of {len(X_test)}")
print(zero_feature)

probe = best_model.predict_proba(vectorizer.transform(["qwertyuiop"]))[0]
print(f"\nPrediction for a nonsense word: {best_model.classes_[probe.argmax()]} (confidence {probe.max():.2f})")
print("-> every unknown input gets this same default guess, below the 0.35 threshold.")

## Phase 5: Justifying the design decisions (experiments)

Each choice in the pipeline (preprocessing steps, feature representation, classifier) is tested against alternatives here, instead of being assumed.

**Method:** repeated stratified 3-fold cross-validation (3 folds × 10 different shuffles = 30 train/test rounds). Each number is the **mean over 30 rounds ± standard deviation**. With only 412 examples, a single split is too noisy to compare options fairly. We report **accuracy** and **macro-F1**, which weights every intent equally, so small intents count as much as large ones.

In [ ]:
from sklearn.model_selection import RepeatedStratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from nltk.stem import PorterStemmer

rcv = RepeatedStratifiedKFold(n_splits=3, n_repeats=10, random_state=RANDOM_STATE)

def evaluate(name, pipeline, X, y=None):
    y = df['intent'] if y is None else y
    r = cross_validate(pipeline, X, y, cv=rcv, scoring=['accuracy', 'f1_macro'])
    return {'Option': name,
            'Accuracy': f"{r['test_accuracy'].mean():.3f} ± {r['test_accuracy'].std():.3f}",
            'Macro-F1': f"{r['test_f1_macro'].mean():.3f}",
            '_acc': r['test_accuracy'].mean()}

def svc_pipeline(vec=None):
    return Pipeline([('tfidf', vec or TfidfVectorizer(ngram_range=(1, 2))),
                     ('clf', LinearSVC(random_state=RANDOM_STATE, max_iter=10000))])

def show(rows):
    t = pd.DataFrame(rows)
    best = t['_acc'].max()
    t['vs. best'] = (t['_acc'] - best).map(lambda d: 'best' if d == 0 else f"{d*100:+.1f} pts")
    return t.drop(columns='_acc')

### 5.1 Preprocessing: which steps actually help?
Features (TF-IDF 1–2 grams) and model (LinearSVC) are held fixed; only the preprocessing changes.

In [ ]:
porter = PorterStemmer()

def basic_clean(t):
    t = t.lower().translate(str.maketrans('', '', string.punctuation))
    return re.sub(r'\s+', ' ', t).strip()

def remove_stopwords(t):
    return ' '.join(w for w in preprocess_text(t).split() if w not in ENGLISH_STOP_WORDS)

def porter_stem(t):
    return ' '.join(porter.stem(w) for w in word_tokenize(basic_clean(t)))

def lemmatize_verbs_too(t):
    return ' '.join(lemmatizer.lemmatize(lemmatizer.lemmatize(w, 'v'), 'n') for w in word_tokenize(basic_clean(t)))

prep_options = {
    'Lowercase only': lambda t: t.lower(),
    '+ remove punctuation, tokenize': lambda t: ' '.join(word_tokenize(basic_clean(t))),
    '+ lemmatize (OUR PIPELINE)': preprocess_text,
    '+ lemmatize + remove stop words': remove_stopwords,
    'Porter stemming instead of lemmatizing': porter_stem,
    'Lemmatize verbs too (POS-aware)': lemmatize_verbs_too,
}
prep_rows = [evaluate(name, svc_pipeline(), df['query'].apply(fn)) for name, fn in prep_options.items()]
emptied = (df['query'].apply(remove_stopwords) == '').sum()
print(f"Stop-word removal turns {emptied} of {len(df)} queries into EMPTY strings, e.g.:")
print(df.loc[df['query'].apply(remove_stopwords) == '', 'query'].head(8).tolist())
show(prep_rows)

**Reading the result:**
- **Lemmatization helps** (about +2 points over no normalization). It merges "fees"/"fee" and "documents"/"document", so fewer distinct words are needed to cover the same meaning.
- **Stop-word removal hurts** (about −5 points), so we deliberately do **not** use it. In this domain, the "stop words" *are* the intent: "**where** is the library" vs "**when** is the library open", or "**who** are **you**". Standard stop-word lists delete exactly those words, and some queries ("who are you", "what can you do") become empty.
- **POS-aware lemmatization and Porter stemming score slightly higher**, about +1 point, because they also reduce verbs ("enrolling" → "enroll"). The gain is within one standard deviation, so it's a candidate improvement, not a clear win.

### 5.2 Feature representation: why TF-IDF, and which n-grams?
Preprocessing (ours) and model (LinearSVC) are held fixed; only the text representation changes.

In [ ]:
X_clean = df['clean_query']
feature_options = {
    'Bag of words (raw counts), unigrams': CountVectorizer(),
    'TF-IDF, unigrams': TfidfVectorizer(),
    'TF-IDF, unigrams + bigrams (OUR PIPELINE)': TfidfVectorizer(ngram_range=(1, 2)),
    'TF-IDF, 1-3 grams': TfidfVectorizer(ngram_range=(1, 3)),
    'TF-IDF, character 2-4 grams': TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 4)),
    'TF-IDF, words 1-2 + characters 2-4': FeatureUnion([
        ('words', TfidfVectorizer(ngram_range=(1, 2))),
        ('chars', TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 4)))]),
}
show([evaluate(name, svc_pipeline(vec), X_clean) for name, vec in feature_options.items()])

**Reading the result:**
- **TF-IDF beats raw counts.** IDF down-weights words that appear in many intents ("what", "is", "college") and up-weights distinctive ones ("tuition", "dorm", "registrar"). That's the job a stop-word list would do, but softly, without deleting the question words that matter.
- **Bigrams did not help on this dataset.** Unigrams alone score about the same, slightly higher but within noise. We expected phrases like "how much" or "where is" to matter. With only ~11 examples per intent, though, most bigrams appear once, so they add features without adding evidence. 1–3 grams are worse for the same reason.
- **Character n-grams give the largest gain** (about +3.5–4 points). They split words into pieces ("enr", "nro", "rol"...), so "enrolling" still overlaps with "enroll", and typos or slang ("okie", "heyy") partly match known words. This directly targets error-analysis cause #1 (out-of-vocabulary inputs).

### 5.3 Classifier: why LinearSVC?
Preprocessing and features (TF-IDF 1–2 grams) are held fixed.

In [ ]:
model_options = {
    'Multinomial Naive Bayes (default)': MultinomialNB(),
    'Multinomial Naive Bayes (alpha=0.1)': MultinomialNB(alpha=0.1),
    'k-Nearest Neighbour (k=1, cosine)': KNeighborsClassifier(n_neighbors=1, metric='cosine'),
    'Logistic Regression': LogisticRegression(max_iter=5000, C=10),
    'LinearSVC (OUR PIPELINE)': LinearSVC(random_state=RANDOM_STATE, max_iter=10000),
}
show([evaluate(name, Pipeline([('tfidf', TfidfVectorizer(ngram_range=(1, 2))), ('clf', clf)]), X_clean)
      for name, clf in model_options.items()])

**Reading the result:**
- **LinearSVC is the best of the five.** Linear SVMs are a standard strong baseline for short-text classification with sparse, high-dimensional TF-IDF features: ~800 features but only ~400 examples. They look for the boundary with the widest margin between intents, which generalizes well from few examples.
- **Naive Bayes is far behind.** It assumes words are independent and is very sensitive to class size, and here 38 classes with 3–20 examples each give it poor probability estimates. Lowering its smoothing (alpha) helps but doesn't close the gap.
- **1-nearest-neighbour** only copies the label of the single most similar training sentence, so one unusual example can mislead it.
- **Why the calibration wrapper:** a plain LinearSVC outputs scores, not probabilities. `CalibratedClassifierCV` turns them into probabilities, which the chatbot needs for its confidence threshold and "not sure" fallback.

### 5.4 Choosing the confidence threshold
Out-of-fold probabilities (every query is scored by a model that never saw it) show the trade-off. A **higher threshold** means fewer wrong answers but more "please rephrase" fallbacks.

In [ ]:
from sklearn.model_selection import cross_val_predict

def calibrated(vec=None, k=2):
    return Pipeline([('tfidf', vec or TfidfVectorizer(ngram_range=(1, 2))),
                     ('clf', CalibratedClassifierCV(LinearSVC(random_state=RANDOM_STATE, max_iter=10000), cv=k))])

y_all = df['intent'].values
oof = cross_val_predict(calibrated(), X_clean, y_all, method='predict_proba',
                        cv=StratifiedKFold(3, shuffle=True, random_state=RANDOM_STATE))
oof_pred = np.array(sorted(set(y_all)))[oof.argmax(1)]
oof_conf = oof.max(1)

rows = []
for t in [0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50]:
    answered = oof_conf >= t
    rows.append({'Threshold': t,
                 'Questions answered': f"{answered.mean():.1%}",
                 'Accuracy of answers given': f"{(oof_pred[answered] == y_all[answered]).mean():.1%}",
                 'Wrong answers (of all questions)': f"{((oof_pred != y_all) & answered).mean():.1%}",
                 'Fallbacks': f"{(~answered).mean():.1%}"})
print(f"Average confidence of CORRECT predictions: {oof_conf[oof_pred == y_all].mean():.2f}")
pd.DataFrame(rows)

**Reading the result:** at **0.35**, the bot gives a wrong answer to only about 1 in 20 questions, and about 93% of the answers it gives are correct. The cost is that it falls back on roughly a third of in-domain questions. We chose to favour **precision over coverage**: for student services, a wrong answer about fees or requirements is worse than "please rephrase or contact the office". Lowering to 0.25 would answer more but roughly triple the wrong answers. Note that calibrated probabilities are spread across 38 intents, so even correct answers average only about 0.47 confidence. That's why the threshold sits well below 0.5.

### 5.5 Robustness: new phrasings and off-topic questions
Cross-validation only tests sentences written in the same style as the training data. Here we test the **final model** (trained on all data) on:
- **45 new phrasings** written by us, none of which appear in `intents.json`, including slang, synonyms and Taglish.
- **20 out-of-scope questions** that no intent should answer. The bot *should* fall back on these.

In [ ]:
UNSEEN = [
    ("hello po", "greeting"), ("good morning", "greeting"), ("yo whats good", "greeting"),
    ("thank you so much", "salutation"), ("ty", "salutation"), ("bye for now", "goodbye"), ("im heading out", "goodbye"),
    ("how much do i pay per sem", "fees"), ("magkano tuition", "fees"), ("how expensive is it to study here", "fees"),
    ("what degrees can i take", "course"), ("do you offer computer engineering", "course"),
    ("where exactly is the campus", "location"), ("whats the school address", "location"),
    ("is there housing for students", "dormitory"), ("can i live on campus", "dormitory"),
    ("when does the library open", "library"), ("where can i borrow books", "library"),
    ("what do i need to enroll", "admission"), ("how can i apply as a freshman", "admission"),
    ("how to apply for scholarship", "scholarship"), ("do you give financial assistance", "scholarship"),
    ("i need a copy of my tor", "document_request"), ("how to get my diploma", "document_request"),
    ("where is the registrar", "office_location"), ("where do i pay my tuition", "office_location"),
    ("when are the finals", "sem"), ("midterm exam schedule", "sem"), ("when is christmas break", "vacation"),
    ("who is the dean of computer studies", "computerhod"), ("whos in charge of the IT department", "computerhod"),
    ("what clubs and orgs are there", "committee"), ("is there a gym", "sports"), ("can i play basketball here", "sports"),
    ("where can i eat", "canteen"), ("what food do they sell", "menu"), ("what should i wear to class", "uniform"),
    ("who built this bot", "creator"), ("what can you help me with", "task"), ("whats your name bot", "name"),
    ("what time does school open", "hours"), ("whats the hotline", "number"), ("any upcoming events", "event"),
    ("do graduates get jobs", "placement"), ("how many students per section", "college intake"),
]
OUT_OF_SCOPE = [
    "what is the weather today", "who won the nba game last night", "how do i cook adobo",
    "can you help me with my math homework", "what is the capital of france", "recommend a good movie",
    "how to fix my laptop", "what time is it", "tell me a joke", "how do i reset my facebook password",
    "what is the price of bitcoin", "where can i buy shoes", "who is the president of the philippines",
    "how to lose weight", "translate hello to japanese", "what is love", "is it going to rain",
    "how old are you", "play some music", "order pizza for me",
]

def robustness(vec_factory, label):
    model = calibrated(vec_factory(), k=3).fit(X_clean, y_all)
    uq = [preprocess_text(q) for q, _ in UNSEEN]
    true = np.array([l for _, l in UNSEEN])
    pu = model.predict_proba(uq)
    pred, conf = model.classes_[pu.argmax(1)], pu.max(1)
    oos_conf = model.predict_proba([preprocess_text(q) for q in OUT_OF_SCOPE]).max(1)
    ok = conf >= CONFIDENCE_THRESHOLD
    summary = {'Features': label,
               'New phrasings: top-1 correct': f"{(pred == true).mean():.1%}",
               'answered correctly': f"{((pred == true) & ok).mean():.1%}",
               'answered WRONG': f"{((pred != true) & ok).mean():.1%}",
               'fell back': f"{(~ok).mean():.1%}",
               'Off-topic correctly rejected': f"{(oos_conf < CONFIDENCE_THRESHOLD).mean():.1%}"}
    details = pd.DataFrame({'query': [q for q, _ in UNSEEN], 'expected': true, 'predicted': pred, 'confidence': conf.round(2)})
    return summary, details, model

word_only = lambda: TfidfVectorizer(ngram_range=(1, 2))
word_char = lambda: FeatureUnion([('words', TfidfVectorizer(ngram_range=(1, 2))),
                                  ('chars', TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 4)))])
s1, d1, m1 = robustness(word_only, 'Words 1-2 (OUR PIPELINE)')
s2, d2, m2 = robustness(word_char, 'Words 1-2 + characters 2-4')
display(pd.DataFrame([s1, s2]))

print("\nNew phrasings our pipeline got wrong:")
display(d1[d1['expected'] != d1['predicted']])

print("Off-topic questions our pipeline answered anyway (confidence >= threshold):")
oos_p = m1.predict_proba([preprocess_text(q) for q in OUT_OF_SCOPE])
display(pd.DataFrame([(q, m1.classes_[p.argmax()], round(p.max(), 2)) for q, p in zip(OUT_OF_SCOPE, oos_p)
                      if p.max() >= CONFIDENCE_THRESHOLD], columns=['query', 'answered as', 'confidence']))

**Reading the result:**
- **The biggest weakness is synonyms.** The model gets only about 58% of new phrasings right, far below its ~81% cross-validation score. Most failures use a word the training data never contains: "**housing**" (dorm), "**gym**" (sports), "**finals**" (exam), "**dean**" (HOD), "**hotline**" (contact number). TF-IDF matches *words*, not *meanings*, so an unseen synonym carries no signal.
- **Off-topic questions are only partly rejected.** About half get a confident wrong answer. "how to fix my laptop" scores high for `document_request` because it shares "how to" and "my" with "how to get **my** documents". The model has no notion of "none of the above"; it can only pick the closest of the 38 intents.
- **Character n-grams improve both robustness measures.** They reduce wrong answers on new phrasings and reject more off-topic questions, but they don't fix synonyms (top-1 accuracy is unchanged). That needs meaning-aware representations, such as word or sentence embeddings, or more training phrasings that include the synonyms.

## Exporting the Final Model for Deployment

Everything evaluated above (the 80/20 split, cross-validation, confusion matrix, error analysis) used a model trained on 80% of the data, so 20% could be held back to honestly test it. For actual deployment, the standard practice is: once you trust the architecture (confirmed above via CV), retrain on **all** available data for maximum real-world coverage, then save that exact trained object to a file.

The cell below does exactly that, and saves two files -- `vectorizer.joblib` and `model.joblib` -- which the Flask backend (`backend/app.py`) loads directly, rather than retraining its own separate copy. This closes the gap between "the model we evaluated" and "the model actually running in the live demo": they become the same trained object, not two independently-trained copies of the same recipe.


In [15]:
import joblib

# Train the FINAL model on all available data (not just the 80% training split --
# the held-out 20% already did its job during evaluation above).
final_vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=1)
X_full_tfidf = final_vectorizer.fit_transform(df["clean_query"])

min_class_count_full = df["intent"].value_counts().min()
final_safe_cv = max(2, min(3, min_class_count_full))

final_base_svc = LinearSVC(random_state=RANDOM_STATE, max_iter=10000)
final_model = CalibratedClassifierCV(final_base_svc, cv=final_safe_cv)
final_model.fit(X_full_tfidf, df["intent"])

joblib.dump(final_vectorizer, "vectorizer.joblib")
joblib.dump(final_model, "model.joblib")

print(f"Exported vectorizer.joblib (vocabulary size: {len(final_vectorizer.vocabulary_)})")
print(f"Exported model.joblib (classes: {len(final_model.classes_)})")
print("\nCopy both files AND askcampus_dialogue.py into backend/ in your repo -- app.py loads them directly.")


Exported vectorizer.joblib (vocabulary size: 824)
Exported model.joblib (classes: 38)

Copy both files into backend/ in your repo -- app.py loads them directly.


### Using the final exported model from here on

`predict_intent()` looks up `best_model` / `vectorizer` fresh on every call (not frozen at definition time), so reassigning them here makes `chatbot_response()` and `run_cli_chatbot()` -- including if you run the CLI chatbot below, right here in Colab -- use the exact same trained model object that was just exported to `model.joblib` / `vectorizer.joblib` and that powers the live web demo. Before this cell, those functions were still using the 80/20-split evaluation model from earlier in the notebook.


In [16]:
best_model = final_model
vectorizer = final_vectorizer

print("best_model and vectorizer now point to the final exported model.")
print("Try it: run_cli_chatbot() in the next cell uses this exact model.")


best_model and vectorizer now point to the final exported model.
Try it: run_cli_chatbot() in the next cell uses this exact model.


## Summary

| Item | Choice | Evidence (Phase 5) |
|---|---|---|
| Dataset | 38 intents, 412 patterns from `intents.json` | Quality check: 6 duplicate patterns, 1 empty pattern, 11 intents with < 8 patterns |
| Preprocessing | Lowercase, punctuation removal, NLTK tokenization, WordNet lemmatization; **no** stop-word removal | Lemmatizing +2 pts; stop-word removal −5 pts |
| Features | TF-IDF, unigrams + bigrams | TF-IDF > raw counts; bigrams ≈ no gain; character n-grams +3.5 pts (recommended next step) |
| Model | LinearSVC + probability calibration | Best of 5 classifiers (~81% CV accuracy vs 56% Naive Bayes) |
| Fallback | Confidence threshold 0.35 | ~93% of given answers correct; ~1 in 3 in-domain questions falls back |
| Dialogue context | `askcampus_dialogue.py`: per-user state, sticky fees topic, course entity + fuzzy matching, yes/no follow-up | Fixes multi-turn failures a single-message classifier can't handle |
| Interface | CLI chat loop + Flask web app (same dialogue module) | |
| Evaluation | Accuracy, Precision, Recall, F1, Confusion Matrix, CV, error analysis, threshold sweep, new-phrasing and off-topic tests | |

**Main weaknesses:** synonyms and new vocabulary (~58% on new phrasings), off-topic questions often answered anyway, overlapping intents (`sem`/`vacation`, `document`/`document_request`), thin intents. **Next steps:** add character n-grams, add synonym phrasings to thin intents, merge overlapping intents, try sentence embeddings for semantic similarity.

## Demo: sample conversations

This scripted run doesn't wait for typing, so **Run all** never hangs, and its output covers the *sample conversations* deliverable. `handled by` shows whether the ML model or the context layer produced each reply. The second part runs two students at once to show their conversations are kept separate.

In [ ]:
demo_bot = AskCampusBot(best_model, vectorizer, response_bank, threshold=CONFIDENCE_THRESHOLD)

def show(state, msg, who="Student A"):
    r = demo_bot.respond(msg, state)
    print(f"{who}: {msg}")
    print(f"AskCampus: {r['reply']}")
    print(f"   [intent={r['intent']}, confidence={r['confidence']:.2f}, handled by: {r['handled_by']}]\n")

student_a = ConversationState()
for msg in [
    "hi",
    "what courses do you offer",
    "how much are the fees",
    "computer science",               # answer to the bot's question (context)
    "what about IT",
    "compute4r science",              # typo -> fuzzy match
    "where is the library",           # topic change; 'is' is NOT read as a course
    "what are the fees for information system",
    "how do i enroll",
    "no",
]:
    show(student_a, msg)

print("=" * 60)
print("Two students at the same time, each with their own conversation state:\n")
student_b, student_c = ConversationState(), ConversationState()
show(student_b, "how much are the fees", who="Student B")
show(student_c, "how much are the fees", who="Student C")
show(student_b, "IT", who="Student B")                 # B gets IT fees
show(student_c, "where is the library", who="Student C")  # C changes topic -> leaves fees mode
show(student_b, "what about computer science", who="Student B")  # B is still in fees mode


### Interactive chat (run this cell by itself for the live demo)

In [ ]:
run_cli_chatbot(show_debug=True)   # show_debug=False hides the intent/confidence line
